# Apache Spark DataFrame Basics Assignment

## Step 1: Start Spark Session

In [11]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, count, sum, min, max, to_timestamp
from pyspark.sql.types import TimestampType

# Create a local Spark session
spark = SparkSession.builder \
    .appName("SparkDataFrameBasics") \
    .getOrCreate()

print("Spark Session initialized:", spark)

Spark Session initialized: <pyspark.sql.session.SparkSession object at 0x000001B5551DEE40>


## Step 2: Load Data

We load the synthetic `dataset.csv` from the `data` directory.

In [12]:
# Load CSV data with header and automatically infer columns types
df = spark.read.csv("../data/dataset.csv", header=True, inferSchema=True)

# Show first 5 rows
print("First 5 rows:")
df.show(5)

# Print Column Names
print("Column Names:", df.columns)

# Display Schema structure
df.printSchema()

# Total record count
print(f"Total raw records count: {df.count()}")

First 5 rows:
+-------+---------+--------------------+---+------------+------+--------+----------------+-----------+------+--------+----------------+-------------------+---------+
|user_id| username|               email|age|subscription|region|    city|product_category|sale_amount| price|store_id|transaction_date|      raw_timestamp|   status|
+-------+---------+--------------------+---+------------+------+--------+----------------+-----------+------+--------+----------------+-------------------+---------+
|   1081|user_1081|user_1081@example...| 30|       Basic|  East|New York|        Clothing|     370.87|429.05|     101|      2026-06-30|30/06/2026 12:00 AM|  Pending|
|   1029|user_1029|                NULL| 27|        Free|  East|New York|        Clothing|     230.11| 264.1|     102|      2026-06-26|2026-06-26 00:00:00|  Pending|
|   1097|user_1097|user_1097@example...| 21|    Standard|  East|New York|  Home & Kitchen|     305.83|333.93|     101|      2026-06-19|2026-06-19 00:00:00|C

## Step 3: Data Cleaning

We will perform:
1. Duplicate removal based on `user_id` and `transaction_date`.
2. Filling missing values (`status` column nulls with `'Unknown'`).
3. Filtering out rows where `email` is null OR `username` is empty.

In [13]:
# 1. Remove duplicate rows based on user_id and transaction_date
df_cleaned = df.dropDuplicates(subset=["user_id", "transaction_date"])
print(f"Count after removing user-date duplicates: {df_cleaned.count()}")

# 2. Handle missing values: Fill status nulls with 'Unknown'
df_cleaned = df_cleaned.fillna(value="Unknown", subset=["status"])

# 3. Remove rows with null email OR empty username
df_cleaned = df_cleaned.filter(col("email").isNotNull() & (col("username") != ""))
print(f"Count after filtering null email and empty username: {df_cleaned.count()}")

Count after removing user-date duplicates: 1191
Count after filtering null email and empty username: 1088


## Step 4: Filter Data

We perform two filter operations:
1. Filtering users with `age` between 18 and 30 (inclusive) with a `'Premium'` subscription.
2. Filtering records from the `'West'` region.

In [14]:
# Filter age 18-30 (inclusive) and subscription 'Premium'
df_premium_young = df_cleaned.filter(
    (col("age") >= 18) & 
    (col("age") <= 30) & 
    (col("subscription") == "Premium")
)
print(f"Count of Premium users aged 18-30: {df_premium_young.count()}")
df_premium_young.show(5)

# Filter for West region
df_west = df_cleaned.filter(col("region") == "West")
print(f"Count of records in West region: {df_west.count()}")

Count of Premium users aged 18-30: 59
+-------+---------+--------------------+---+------------+------+-----------+----------------+-----------+------+--------+----------------+-------------------+---------+
|user_id| username|               email|age|subscription|region|       city|product_category|sale_amount| price|store_id|transaction_date|      raw_timestamp|   status|
+-------+---------+--------------------+---+------------+------+-----------+----------------+-----------+------+--------+----------------+-------------------+---------+
|   1001|user_1001|user_1001@example...| 22|     Premium|  East|   New York|     Electronics|     290.63|296.15|     101|      2026-07-09|2026-07-09 00:00:00|Cancelled|
|   1001|user_1001|user_1001@example...| 27|     Premium|  West|Los Angeles|          Sports|     312.09|372.81|     103|      2026-07-18|2026-07-18 00:00:00|  Pending|
|   1003|user_1003|user_1003@example...| 27|     Premium|  West|Los Angeles|        Clothing|     479.79|  NULL|     

## Step 5: Transform Data

We cast the `raw_timestamp` column to a `TimestampType` and rename it to `event_time`.

In [15]:
# Cast raw_timestamp to timestamp type and rename to event_time
df_transformed = df_cleaned \
    .withColumn("raw_timestamp", col("raw_timestamp").cast(TimestampType())) \
    .withColumnRenamed("raw_timestamp", "event_time")

df_transformed.select("user_id", "username", "event_time").show(5)
df_transformed.printSchema()

+-------+---------+-------------------+
|user_id| username|         event_time|
+-------+---------+-------------------+
|   1000|user_1000|2026-06-18 00:00:00|
|   1000|user_1000|2026-06-20 00:00:00|
|   1000|user_1000|2026-06-23 00:00:00|
|   1000|user_1000|2026-06-28 00:00:00|
|   1000|user_1000|2026-07-08 00:00:00|
+-------+---------+-------------------+
only showing top 5 rows
root
 |-- user_id: integer (nullable = true)
 |-- username: string (nullable = true)
 |-- email: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- subscription: string (nullable = true)
 |-- region: string (nullable = true)
 |-- city: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- sale_amount: double (nullable = true)
 |-- price: double (nullable = true)
 |-- store_id: integer (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- status: string (nullable = false)



## Step 6: Aggregations & Grouping

We will run three aggregate/group operations:
1. Calculate multiple metrics (min, max, mean) for `price` column using `.agg()`.
2. Filter for `'West'` region, then group by `product_category` to get the average `sale_amount`.
3. Group by `city` to find counts, filtering for counts greater than 100.

In [16]:
# 1. Calculate price statistics
print("Price Stats (Min, Max, Avg):")
df_cleaned.agg(
    min("price").alias("min_price"),
    max("price").alias("max_price"),
    avg("price").alias("mean_price")
).show()

# 2. Avg sale amount per product category in the West region
print("Avg Sale Amount by Category (West region):")
df_cleaned.filter(col("region") == "West") \
    .groupBy("product_category") \
    .agg(avg("sale_amount").alias("avg_sale_amount")) \
    .show()

# 3. Cities with record counts greater than 100
print("Cities with count > 100:")
df_cleaned.groupBy("city") \
    .count() \
    .filter(col("count") > 100) \
    .show()

Price Stats (Min, Max, Avg):
+---------+---------+------------------+
|min_price|max_price|        mean_price|
+---------+---------+------------------+
|     9.63|   585.95|254.27383744855996|
+---------+---------+------------------+

Avg Sale Amount by Category (West region):
+----------------+------------------+
|product_category|   avg_sale_amount|
+----------------+------------------+
|  Home & Kitchen|234.31239999999994|
|          Sports|244.61079207920795|
|     Electronics|248.71259615384616|
|        Clothing| 267.6641573033708|
|           Books| 252.3475630252101|
+----------------+------------------+

Cities with count > 100:
+-----------+-----+
|       city|count|
+-----------+-----+
|Los Angeles|  251|
|   New York|  387|
+-----------+-----+



## Step 7: Build a Simple Pipeline

Combine steps to:
1. Filter out all duplicates from original dataset.
2. Fill missing prices with `0`.
3. Group by `store_id` to compute the `total_revenue`.
4. Save the results as a CSV file.

In [ ]:
# Complete Pipeline execution
df_pipeline = df \
    .dropDuplicates() \
    .fillna({"price": 0}) \
    .groupBy("store_id") \
    .agg(sum("price").alias("total_revenue")) \
    .sort("store_id")

print("Pipeline Total Revenue results:")
df_pipeline.show()

# Export results as pandas to save single CSV in output folder
results_pd = df_pipeline.toPandas()
results_pd.to_csv("../output/results.csv", index=False)
print("Saved results table successfully to output/results.csv")

Pipeline Total Revenue results:
+--------+-----------------+
|store_id|    total_revenue|
+--------+-----------------+
|     101|79699.09999999999|
|     102|88791.44000000002|
|     103|79588.55000000002|
|     104|89696.75999999997|
+--------+-----------------+

Saved results table successfully to output/results.csv


: 